In [ ]:
pip install pdfplumber

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
import re
import pdfplumber
import pandas as pd
from dotenv import load_dotenv
from pathlib import Path
import shutil

# Adds the folder above the current one to the search path
sys.path.append(os.path.abspath(".."))
from src.parser import extract_data_from_searchable_pdf, extract_source_b, extract_source_c_csv

# Load environment variables
load_dotenv()

In [ ]:
# base and source paths
base_dir = Path(os.getenv("data_dir"))
OUTPUT_DIR = base_dir / "processed"
source_a_dir = Path(os.getenv("platform_1"))
output_source_a_csv = OUTPUT_DIR / "portfolio_platform1_input.csv"
source_b_path = base_dir / os.getenv("platform_2")
output_source_b_csv = OUTPUT_DIR / "portfolio_platform2_input.csv"
source_c_path = base_dir / os.getenv("platform_3")
output_source_c_csv = OUTPUT_DIR / "portfolio_platform3_input.csv"

In [ ]:
# --- Get source A (pdf files) and convert it to csv data ---
#****** before running this cell please read description.txt file ******
results = []
for pdf_file in source_a_dir.glob("*.pdf"):
    results.append(extract_data_from_searchable_pdf(pdf_file))

df = pd.DataFrame(results)
df.to_csv(output_source_a_csv, index=False)
print(f"Expert extraction complete. File saved: {output_source_a_csv}")

In [ ]:
# --- Get source B (pdf files) and convert it to csv data ---
archive_dir = source_b_path / "original_files_archive"
archive_dir.mkdir(exist_ok=True)
all_results = []

# 1. RESTORE: Move everything from archive back to main folder(so that old files can also be processed)
for archived_file in archive_dir.glob("*.pdf"):
    shutil.move(str(archived_file), str(source_b_path / archived_file.name))

all_results = []

# 2. PROCESS
for file_path in source_b_path.glob("*.pdf"):
    filename = file_path.name
    
    # Process only the specific targets
    if "Contract-note" in filename or "Corporate-actions" in filename:
        try:
            with pdfplumber.open(file_path) as pdf:
                text = pdf.pages[0].extract_text() or ""
                result = extract_source_b(file_path, text)
                all_results.append(result)
        except Exception as e:
            print(f"Error in {filename}: {e}")
    
    # 3. ARCHIVE
    shutil.move(str(file_path), str(archive_dir / filename))

# 4. SAVE
if all_results:
    df_b = pd.DataFrame(all_results)
    df_b.to_csv(output_source_b_csv, index=False)
    print(f"Done! Processed {len(all_results)} files.")

In [ ]:
# --- Get source C files (yearly csv) and combine them alltogether---
all_source_c_data = []

for file_path in source_c_path.glob("*.csv"):
    print(f"Processing CSV: {file_path.name}")
    try:
        file_data = extract_source_c_csv(file_path)
        all_source_c_data.extend(file_data)
    except Exception as e:
        print(f"Error in {file_path.name}: {e}")

# Save the combined results
if all_source_c_data:
    df_final_c = pd.DataFrame(all_source_c_data)
    df_final_c.to_csv(output_source_c_csv, index=False)
    print(f"Success! Combined {len(all_source_c_data)} rows into {output_source_c_csv}")